# 📧 Email Triage RL Training Demo (GRPO)

This notebook demonstrates how to train a small LLM (e.g., Llama-3-8B) on the Email Triage OpenEnv using GRPO with `trl`.

In [ ]:
!pip install trl peft transformers datasets accelerate openenv-core

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOTrainer, GRPOConfig
from client import EmailTriageEnvClient

# Initialize environment client
env = EmailTriageEnvClient("http://localhost:8000")

# Load model and tokenizer
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

In [ ]:
def env_reward_function(completions, prompts):
    """
    Custom reward function that queries the OpenEnv for a true RL signal.
    """
    rewards = []
    for completion, prompt in zip(completions, prompts):
        # Reset env on each completion path (simplified for demo)
        env.reset(task_name="medium")
        
        # Simplified JSON parsing from LLM output
        import json
        try:
            action = json.loads(completion.strip())
        except Exception:
            action = {"action_type": "noop"}
            
        # Step env to get reward
        result = env.step(action)
        rewards.append(result["reward"])
        
    return rewards

In [ ]:
# GRPO Training config
config = GRPOConfig(
    output_dir="./email-agent-grpo",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    use_peft=True,  # Use LoRA for efficient training
)

# Create dummy dataset based on env data
from datasets import Dataset
import json
with open("data/emails.json", "r") as f:
    emails = json.load(f)

train_data = []
for e in emails[:100]:
    prompt = f"From: {e['sender']}\nSubject: {e['subject']}\nBody: {e['body']}\n\nRespond with JSON action."
    train_data.append({"prompt": prompt})
dataset = Dataset.from_list(train_data)

# Initialize Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=env_reward_function,
    args=config,
    train_dataset=dataset,
)

# Train!
trainer.train()